# MODEL TRAINING 

### autoencoder

### SVM - support vector machine 


x_train = is ALL THE UNLABELLED DATA

x test + y test - ALL DATA INCLUDING LABELLED which acts as a control

In [41]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import MinMaxScaler


In [ ]:
# Define normalized features created during preprocessing
combined_df = pd.read_csv('df_processed.csv')
feature_cols = ['z_igf1', 'z_piiinp', 'z_biomarker_ratio', 'age', 'sex']

# Separate baseline unlabelled data (-1) from positive washout samples (1)
baseline_df = combined_df[combined_df['target'] == -1].dropna(subset=feature_cols)

admin_df = combined_df[combined_df['target'] == 1].dropna(subset=feature_cols)

X_baseline = baseline_df[feature_cols].values.astype(np.float32)
X_admin = admin_df[feature_cols].values.astype(np.float32)


In [43]:
class AutoencoderSVMModel:
    """
    Pure model class that fits the Autoencoder + One-Class SVM pipeline 
    and outputs unranked continuous risk scores.
    """
    def __init__(self, input_dim, latent_dim=3, nu=0.02, epochs=50, batch_size=64):
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.nu = nu
        self.epochs = epochs
        self.batch_size = batch_size
        
        # Build models
        self.encoder, self.autoencoder = self._build_autoencoder()
        self.svm = OneClassSVM(kernel='rbf', gamma='scale', nu=self.nu)
        self.scaler = MinMaxScaler(feature_range=(0, 1))

    def _build_autoencoder(self):
        encoder_input = layers.Input(shape=(self.input_dim,))
        x = layers.Dense(16, activation='relu')(encoder_input)
        x = layers.Dense(8, activation='relu')(x)
        latent = layers.Dense(self.latent_dim, activation='linear')(x)
        encoder = models.Model(encoder_input, latent, name="Encoder")

        decoder_input = layers.Input(shape=(self.latent_dim,))
        x = layers.Dense(8, activation='relu')(decoder_input)
        x = layers.Dense(16, activation='relu')(x)
        reconstructed = layers.Dense(self.input_dim, activation='linear')(x)
        decoder = models.Model(decoder_input, reconstructed, name="Decoder")

        autoencoder = models.Model(encoder_input, decoder(encoder(encoder_input)), name="Autoencoder")
        autoencoder.compile(optimizer='adam', loss='mse')
        return encoder, autoencoder

    def _extract_latent_and_mse(self, X): # extract the mean squared error (reconstruction error) and latent coords.
        X_pred = self.autoencoder.predict(X, verbose=0)
        X_latent = self.encoder.predict(X, verbose=0)
        mse = np.mean(np.square(X - X_pred), axis=1, keepdims=True)
        return mse, np.hstack([X, X_latent, mse]) # stacks the arrays columnwise (123) + (456) = [123456]

    def fit(self, X_baseline):

        """Fit the Autoencoder and OC- SVM on clean baseline feature array."""
        self.autoencoder.fit(
            X_baseline, X_baseline,
            epochs=self.epochs,
            batch_size=self.batch_size,
            verbose=0
        )
        
        mse_base, combined_base = self._extract_latent_and_mse(X_baseline)
        self.svm.fit(combined_base)
        
        svm_dist_base = self.svm.decision_function(combined_base)
        raw_baseline_metrics = np.column_stack([mse_base.flatten(), -svm_dist_base])
        self.scaler.fit(raw_baseline_metrics)
        self.mse_base_ = mse_base
        return self

    def score_samples(self, df, feature_cols, id_col='sample_id'):
        """
        Calculates raw MSE, SVM distances, and risk scores, returning
        a structured DataFrame attached to sample IDs.
        """
        # Ensure we drop NaNs consistently so array indices match the dataframe rows
        df_clean = df.dropna(subset=feature_cols).copy()
        X = df_clean[feature_cols].values.astype(np.float32)

        # Extract features and raw metrics
        mse, combined_features = self._extract_latent_and_mse(X)
        svm_dist = self.svm.decision_function(combined_features)

        # Scale metrics based on baseline calibration
        raw_metrics = np.column_stack([mse.flatten(), -svm_dist])
        scaled_metrics = self.scaler.transform(raw_metrics)

        ae_risk = scaled_metrics[:, 0]
        svm_risk = scaled_metrics[:, 1]
        composite_score = 0.5 * ae_risk + 0.5 * svm_risk

        # Build output dataframe attached to IDs
        scores_df = pd.DataFrame({
            id_col: df_clean[id_col].values,
            'dataset_source': df_clean['dataset_source'].values if 'dataset_source' in df_clean.columns else 'unknown',
            'target': df_clean['target'].values if 'target' in df_clean.columns else np.nan,
            'mse_raw': mse.flatten(),
            'svm_dist_raw': svm_dist,
            'ae_risk_scaled': ae_risk,
            'svm_risk_scaled': svm_risk,
            'composite_doping_score': composite_score
        })

        return scores_df

### Sort the output of both SVM and AE so that the result scores are on the same MinMaxScale (0-1)

In [44]:
def rank_athletes(df, scores_dict, id_col='sample_id'):
    """
    Standalone function that takes the input dataframe and the model scores, 
    merges them, and generates the final ordered leaderboard.
    """
    results_df = df[[id_col]].copy()
    if 'dataset_source' in df.columns:
        results_df['dataset_source'] = df['dataset_source']
    if 'target' in df.columns:
        results_df['target'] = df['target']
        
    results_df['ae_risk_scaled'] = scores_dict['ae_risk_scaled']
    results_df['svm_risk_scaled'] = scores_dict['svm_risk_scaled']
    results_df['composite_doping_score'] = scores_dict['composite_doping_score']
    
    # Generate Rank (1 = Highest Risk)
    results_df['doping_rank'] = results_df['composite_doping_score'].rank(
        ascending=False, method='min'
    ).astype(int)
    
    return results_df.sort_values(by='doping_rank', ascending=True).reset_index(drop=True)

## CALL THE WORKFLOW AND USE THE ORIGONAL DATA ON IT    

In [46]:
# Feature columns used for modeling
feature_cols = ['z_igf1', 'z_piiinp', 'z_biomarker_ratio', 'age', 'sex']

# 1. TRAIN SET: Extract ONLY baseline controls (target == -1)
train_df = combined_df[combined_df['target'] == -1].dropna(subset=feature_cols)
X_train = train_df[feature_cols].values.astype(np.float32)

# 2. FIT MODEL: Train Autoencoder & SVM strictly on baseline controls
model = AutoencoderSVMModel(input_dim=len(feature_cols))
model.fit(X_train)

# 3. SCORE / TEST SET: Pass ALL data through to detect anomalies
scores_df = model.score_samples(combined_df, feature_cols=feature_cols, id_col='sample_id')

In [52]:
scores_df[scores_df['sample_id'].astype(str).str.startswith('RNAG')]


,sample_id,dataset_source,target,mse_raw,svm_dist_raw,ae_risk_scaled,svm_risk_scaled,composite_doping_score


In [57]:
# 1. View a sample of actual sample IDs across the combined dataset
print(combined_df['sample_id'].head(15))

# 2. Check sample IDs grouped by dataset source to see their format
print(combined_df.groupby('dataset_source')['sample_id'].first())

# 3. Search for a partial string (case-insensitive) if you are looking for a specific study/group
print(combined_df[combined_df['sample_id'].str.contains('RN', case=False, na=False)]['sample_id'].head())

0      394626
1      925536
2     256802V
3     5805591
4     5805878
5      961763
6      973658
7      973797
8     8214977
9      747685
10     964962
11     973812
12    5805477
13    5805703
14    8214724
Name: sample_id, dtype: str
dataset_source
df1_endocrine     394626
df2_daegu         528649
df3_bio           399445
df4_gh_washout         6
Name: sample_id, dtype: str
Series([], Name: sample_id, dtype: str)


### Visualise the models 

In [ ]:
class VisualiseBofa:
    def __init__(self, input_data):
        self.data = input_data
        return self 
    def 